# 🥉 Bronze Layer – Raw Ingestion: 
- Load all CSV files. 
- Understand common columns between files. 
- Keep raw values unchanged for traceability. 
- Store as Bronze staging tables/files. 


In [1]:
import pandas as pd
from pathlib import Path

## Load Data

In [9]:
# Path to the folder containing all raw CSV files
raw_path = Path("../datasets/raw")

# Get all CSV files from the raw dataset folder
csv_files = raw_path.glob("*.csv")

# List to store DataFrames from all CSV files
all_data = []

# Loop through each CSV file
for file in csv_files:

    # Read the current CSV file
    df = pd.read_csv(file)

    # Extract file name without extension
    # Example: amsterdam_weekdays
    file_name = file.stem

    # Split file name into city and day type
    # Result: city = "amsterdam", day_type = "weekdays"
    city, day_type = file_name.rsplit("_", 1)

    # Add city column
    df["city"] = city.capitalize()

    # Add day type column
    df["day_type"] = day_type.capitalize()

    # Store the DataFrame in the list
    all_data.append(df)

# Merge all DataFrames into one DataFrame
bronze_df = pd.concat(all_data, ignore_index=True)

# Display first five rows
print(bronze_df.head())

   Unnamed: 0     realSum     room_type  room_shared  room_private  \
0           0  194.033698  Private room        False          True   
1           1  344.245776  Private room        False          True   
2           2  264.101422  Private room        False          True   
3           3  433.529398  Private room        False          True   
4           4  485.552926  Private room        False          True   

   person_capacity  host_is_superhost  multi  biz  cleanliness_rating  ...  \
0              2.0              False      1    0                10.0  ...   
1              4.0              False      0    0                 8.0  ...   
2              2.0              False      0    1                 9.0  ...   
3              4.0              False      0    1                 9.0  ...   
4              2.0               True      0    0                10.0  ...   

       dist  metro_dist  attr_index  attr_index_norm  rest_index  \
0  5.022964    2.539380   78.690379       

In [10]:
# Display number of rows and columns
print(bronze_df.shape)

(51707, 22)


## Save Bronze Dataset

In [11]:
# Path to save processed files
output_path = Path("../datasets/processed")

# Create the folder if it doesn't already exist
output_path.mkdir(parents=True, exist_ok=True)

# Output file name
output_file = output_path / "bronze_airbnb.csv"

# Save the Bronze dataset
bronze_df.to_csv(output_file, index=False)

print("Bronze layer created successfully!")

Bronze layer created successfully!


# Connect With Sqllite

In [12]:
import sqlite3

In [13]:
# Connect to the SQLite database
connection = sqlite3.connect("../database/airbnb_dwh.db")

# Load the Bronze DataFrame into the Bronze table
bronze_df.to_sql(
    name="bronze_airbnb",
    con=connection,
    if_exists="replace",
    index=False)

# Close the database connection
connection.close()

print("Bronze layer loaded successfully into SQLite.")

Bronze layer loaded successfully into SQLite.


# Connect With SSMS

In [15]:
from sqlalchemy import create_engine
import urllib

params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=DESKTOP-EKQ2K2R;"
    "DATABASE=Airbnb_DWH;"
    "Trusted_Connection=yes;"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [16]:
bronze_df.to_sql(
    name="bronze_airbnb",
    con=engine,
    if_exists="replace",
    index=False
)

27

# 🥉 Bronze Layer

## Objective
The Bronze Layer is responsible for ingesting the raw Airbnb datasets into the Data Warehouse without applying any data cleaning or transformations. The goal is to preserve the original data while preparing it for the next processing stage.

---

## Steps Performed

### 1. Load Raw Data
- Read all Airbnb CSV files from the `datasets/raw` directory.
- Each file represents a specific European city and day type (Weekdays or Weekends).

### 2. Merge All Files
- Combined the 20 CSV files into a single DataFrame using `pandas.concat()`.

### 3. Add Metadata
To identify the source of each record, two metadata columns were added:

- **city** → Extracted from the file name.
- **day_type** → Indicates whether the listing belongs to weekdays or weekends.

Example:

| File Name | City | Day Type |
|-----------|------|----------|
| amsterdam_weekdays.csv | Amsterdam | Weekdays |
| paris_weekends.csv | Paris | Weekends |

### 4. Save the Bronze Dataset
The merged dataset was saved as:

```
datasets/processed/bronze_airbnb.csv
```

### 5. Load into SQL Server
Connected to SQL Server and loaded the Bronze dataset into the Data Warehouse.

Database:

```
Airbnb_DWH
```

Table:

```
bronze_airbnb
```

The table was created using:

- `pandas.to_sql()`
- `if_exists="replace"`

to ensure the pipeline is idempotent and can be executed multiple times without creating duplicate data.

---

## Bronze Layer Output

- Raw merged dataset
- Metadata columns added (`city`, `day_type`)
- Bronze CSV file
- Bronze table stored in SQL Server

---

## Notes

No data cleaning or transformations were applied in this layer.

The dataset remains as close as possible to the original source and serves as the input for the Silver Layer.